# AVENIQ SENTINEL // PHASE 3.1: CHRONOLOGICAL TEMPORAL SPLITTING
## Leakage-Free Temporal Partitioning & Unseen Attack Detection

**Product:** Aveniq Sentinel by Aveniq  
**Evaluation Architecture:** Strict Day-Based Chronological Partitioning  
**Training Period:** Monday - Wednesday  
**Untouched Test Period:** Thursday - Friday  

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from neuralshield.ml.temporal import (
    TemporalSplitter,
    KNOWN_TRAINING_ATTACKS,
    UNSEEN_TEST_ATTACKS,
)
from config.settings import SAMPLE_DATA_DIR

print('=== TEMPORAL SPLITTING ENGINE INITIALIZED ===')
print(f'Known Training Attacks (Mon-Wed): {sorted(list(KNOWN_TRAINING_ATTACKS))}')
print(f'Unseen Test Attacks (Thu-Fri): {sorted(list(UNSEEN_TEST_ATTACKS))}')

=== TEMPORAL SPLITTING ENGINE INITIALIZED ===
Known Training Attacks (Mon-Wed): ['DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'SSH-Patator']
Unseen Test Attacks (Thu-Fri): ['Bot', 'DDoS', 'Infiltration', 'PortScan', 'Web Attack  Brute Force', 'Web Attack  Sql Injection', 'Web Attack  XSS', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [ ]:
# Load active dataset fixture
sample_parquet = SAMPLE_DATA_DIR / 'cic_ids2017_sample.parquet'
df = pd.read_parquet(sample_parquet)

splitter = TemporalSplitter(val_fraction_of_train=0.20)
split = splitter.split_single_dataframe_chronologically(
    df,
    train_ratio=0.60,
    val_ratio=0.15,
    test_ratio=0.25,
)

print('=== CHRONOLOGICAL SPLIT TELEMETRY ===')
print(f'Train Partition: {len(split.train_df):,} flows (60%)')
print(f'Val Partition:   {len(split.val_df):,} flows (15%)')
print(f'Test Partition:  {len(split.test_df):,} flows (25%)')
print(f'Unseen Attack Classes in Test: {split.unseen_test_classes}')

=== CHRONOLOGICAL SPLIT TELEMETRY ===
Train Partition: 903 flows (60%)
Val Partition:   225 flows (15%)
Test Partition:  377 flows (25%)
Unseen Attack Classes in Test: []


In [ ]:
# Separate test set into Known-Attack evaluation vs Novel/Unseen Anomaly evaluation
known_test, unseen_test = splitter.partition_known_vs_unseen_test(
    split.test_df,
    split.train_classes,
    label_col='Label',
)
print(f'Known-Class Test Records: {len(known_test):,}')
print(f'Unseen-Class (Zero-Day) Test Records: {len(unseen_test):,}')
print('Guaranteed Zero Train-Test Overlap.')

Known-Class Test Records: 377
Unseen-Class (Zero-Day) Test Records: 0
Guaranteed Zero Train-Test Overlap.
